# Challenge

Write code that reads the JSONL files from the CLOY data folder and creates dataframe (indexed by datetime, similar to Week 3). Then, using timeseries operations in pandas generate a graph for showing the number of posts / comments over time (daily, weekly, monthly). 

This is my pseudocode that I wrote out:


1. Read JSONL
2. For every line, call json.loads to save as a separate object (Create an updating dictionary, Extract the created_utc value and convert it to a datetime object - should it be timezone aware? If timestamp is new, create new key and increment value by 1; if timestamp is not new, update the original value corresponding to the key by adding 1 
3. We have one dictionary for comments and one dictionary for submissions   

4. Create DFs: Turn dictionaries into dfs and read the keys as index, rename the column as count   
a. Plot by day (X axis = years, Y axis = count; Don’t need to group by but just a line plot connecting each day).  
b. Plot by week / year.   
Resampling (I had to look this up): weekly_df = daily_df.resample('W').sum(), or yearly_df = daily_df.resample(“YE”).sum() 

**NOTE**: I asked Han if i should try to take the timezones into account and I agree that as long as the time is consistent within the dataset, it should be fine.

In [1]:
import json

Let's first figure out how to convert an integer to a datetime object.

In [10]:
from datetime import datetime
timestamp = 1583764623
dt_utc = datetime.fromtimestamp(timestamp)
dt_utc

datetime.datetime(2020, 3, 9, 10, 37, 3)

In [14]:
dt_utc.date()
# making a fake dictionary

dct = {}
dct[dt_utc] = 3

dct

{datetime.datetime(2020, 3, 9, 10, 37, 3): 3}

Can we make this into a DF?

In [ ]:
import pandas as pd
df = pd.DataFrame.from_dict(dct, orient='index') # i got this line from online
df

,0
2020-03-09 10:37:03,3


Maybe we don't want the exact times...? So should we just extract .day() like:

In [19]:
dct = {}
dct[dt_utc.date()] = 3
df = pd.DataFrame.from_dict(dct, orient='index') # i got this line from online
df

,0
2020-03-09,3


Let's start by reading just one cloy_comments_2020.jsonl.

In [ ]:
temp_counts = {}
with open ('/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/cloy_comments_2020.jsonl', 'r', encoding = 'utf-8') as f:
    for line in f:
        data = json.loads(line) # to make sure each json line is handled as a vector
        utc = data['created_utc']
        dt_utc = datetime.fromtimestamp(utc).date()
        temp_counts[dt_utc] = temp_counts.get(dt_utc, 0) + 1   # this checks if the key exists and returns 0 if missing, then add one

In [21]:
temp_counts

{datetime.date(2020, 3, 9): 2,
 datetime.date(2020, 3, 10): 1,
 datetime.date(2020, 3, 11): 4,
 datetime.date(2020, 3, 12): 1,
 datetime.date(2020, 3, 13): 1,
 datetime.date(2020, 3, 14): 1,
 datetime.date(2020, 3, 15): 2,
 datetime.date(2020, 3, 16): 5,
 datetime.date(2020, 3, 17): 4,
 datetime.date(2020, 3, 18): 13,
 datetime.date(2020, 3, 19): 13,
 datetime.date(2020, 3, 20): 2,
 datetime.date(2020, 3, 21): 1,
 datetime.date(2020, 3, 22): 3,
 datetime.date(2020, 3, 23): 1,
 datetime.date(2020, 3, 24): 2,
 datetime.date(2020, 3, 25): 6,
 datetime.date(2020, 3, 26): 6,
 datetime.date(2020, 3, 27): 6,
 datetime.date(2020, 3, 28): 5,
 datetime.date(2020, 3, 29): 5,
 datetime.date(2020, 3, 30): 18,
 datetime.date(2020, 3, 31): 14,
 datetime.date(2020, 4, 1): 23,
 datetime.date(2020, 4, 2): 5,
 datetime.date(2020, 4, 3): 1,
 datetime.date(2020, 4, 4): 4,
 datetime.date(2020, 4, 5): 4,
 datetime.date(2020, 4, 6): 5,
 datetime.date(2020, 4, 7): 2,
 datetime.date(2020, 4, 8): 2,
 datetime.da

Yay that worked, time to run it on everything. To do this, I further separated the data into 2 separate folders.

COMMENTS:

In [39]:
from pathlib import Path

comments_path = Path("/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/comments")
comments = {}

for jsonl in comments_path.iterdir():
    if jsonl.is_file():  
        print(jsonl)
        with open (jsonl, 'r', encoding = 'utf-8') as f:
            for line in f:
                data = json.loads(line) # to make sure each json line is handled as a vector
                utc = data['created_utc']
                dt_utc = datetime.fromtimestamp(utc).date()
                comments[dt_utc] = comments.get(dt_utc, 0) + 1   # this checks if the key exists and returns 0 if missing, then add one
       

/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/comments/cloy_comments_2025.jsonl
/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/comments/cloy_comments_2023.jsonl
/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/comments/cloy_comments_2021.jsonl
/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/comments/cloy_comments_2026.jsonl
/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/comments/cloy_comments_2024.jsonl
/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/comments/cloy_comments_2019.jsonl
/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/comments/cloy_comments_2020.jsonl
/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/comments/cloy_comments_2022.jsonl


In [27]:
len(comments)

2040

In [38]:
comments

{datetime.date(2024, 12, 31): 21,
 datetime.date(2025, 1, 1): 9,
 datetime.date(2025, 1, 2): 26,
 datetime.date(2025, 1, 3): 25,
 datetime.date(2025, 1, 4): 17,
 datetime.date(2025, 1, 5): 21,
 datetime.date(2025, 1, 6): 37,
 datetime.date(2025, 1, 7): 11,
 datetime.date(2025, 1, 8): 8,
 datetime.date(2025, 1, 9): 20,
 datetime.date(2025, 1, 10): 16,
 datetime.date(2025, 1, 11): 19,
 datetime.date(2025, 1, 12): 4,
 datetime.date(2025, 1, 13): 17,
 datetime.date(2025, 1, 14): 41,
 datetime.date(2025, 1, 15): 22,
 datetime.date(2025, 1, 16): 13,
 datetime.date(2025, 1, 17): 36,
 datetime.date(2025, 1, 18): 11,
 datetime.date(2025, 1, 19): 9,
 datetime.date(2025, 1, 20): 4,
 datetime.date(2025, 1, 21): 2,
 datetime.date(2025, 1, 22): 10,
 datetime.date(2025, 1, 23): 32,
 datetime.date(2025, 1, 24): 8,
 datetime.date(2025, 1, 25): 17,
 datetime.date(2025, 1, 26): 54,
 datetime.date(2025, 1, 27): 40,
 datetime.date(2025, 1, 28): 50,
 datetime.date(2025, 1, 29): 21,
 datetime.date(2025, 1, 3

In [29]:
submissions_path = Path('/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks-new/data/CLOY/submissions')
submissions = {}

for jsonl in submissions_path.iterdir():
    if jsonl.is_file():  
        with open (jsonl, 'r', encoding = 'utf-8') as f:
            for line in f:
                data = json.loads(line) # to make sure each json line is handled as a vector
                utc = data['timestamp']
                dt_utc = datetime.fromtimestamp(utc).date()
                submissions[dt_utc] = submissions.get(dt_utc, 0) + 1   # this checks if the key exists and returns 0 if missing, then add one

In [31]:
len(submissions)

1400

In [37]:
submissions

{datetime.date(2025, 1, 1): 1,
 datetime.date(2025, 1, 2): 3,
 datetime.date(2025, 1, 3): 2,
 datetime.date(2025, 1, 4): 1,
 datetime.date(2025, 1, 5): 3,
 datetime.date(2025, 1, 6): 2,
 datetime.date(2025, 1, 8): 1,
 datetime.date(2025, 1, 9): 3,
 datetime.date(2025, 1, 10): 3,
 datetime.date(2025, 1, 11): 4,
 datetime.date(2025, 1, 12): 1,
 datetime.date(2025, 1, 13): 3,
 datetime.date(2025, 1, 14): 3,
 datetime.date(2025, 1, 15): 1,
 datetime.date(2025, 1, 16): 2,
 datetime.date(2025, 1, 17): 2,
 datetime.date(2025, 1, 18): 2,
 datetime.date(2025, 1, 19): 1,
 datetime.date(2025, 1, 20): 2,
 datetime.date(2025, 1, 21): 1,
 datetime.date(2025, 1, 22): 2,
 datetime.date(2025, 1, 25): 1,
 datetime.date(2025, 1, 26): 5,
 datetime.date(2025, 1, 27): 1,
 datetime.date(2025, 1, 28): 6,
 datetime.date(2025, 1, 29): 2,
 datetime.date(2025, 1, 30): 5,
 datetime.date(2025, 1, 31): 4,
 datetime.date(2025, 2, 1): 1,
 datetime.date(2025, 2, 2): 1,
 datetime.date(2025, 2, 3): 2,
 datetime.date(2025

## Creating DFs

In [32]:
comments_df = pd.DataFrame.from_dict(comments, orient='index')
submissions_df = pd.DataFrame.from_dict(submissions, orient='index')

In [33]:
comments_df.head()

,0
2024-12-31,21
2025-01-01,9
2025-01-02,26
2025-01-03,25
2025-01-04,17


In [36]:
full_df = pd.merge(comments_df, submissions_df, left_index=True, right_index=True)

In [35]:
full_df.head()

,0_x,0_y
2025-01-01,9,1
2025-01-02,26,3
2025-01-03,25,2
2025-01-04,17,1
2025-01-05,21,3


In [40]:
full_df = full_df.rename(columns={'0_x': 'comments-counts', '0_y': 'submissions-counts'})
full_df

,comments-counts,submissions-counts
2025-01-01,9,1
2025-01-02,26,3
2025-01-03,25,2
2025-01-04,17,1
2025-01-05,21,3
...,...,...
2022-12-24,25,1
2022-12-25,11,3
2022-12-26,13,5
2022-12-27,10,1


Last thing, sort by index.

In [41]:
full_df = full_df.sort_index()
full_df.head()

,comments-counts,submissions-counts
2020-03-11,4,1
2020-03-12,1,1
2020-03-15,2,1
2020-03-16,5,2
2020-03-18,13,1


Great!

## Plotting

In [42]:
import matplotlib